In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import pandas as pd
import numpy as np
import ast
import time

In [ ]:
products_df = pd.read_csv('products.csv')
queries_df = pd.read_csv('queries_train.csv')

## Stage 1: Extraction

In [ ]:
import os
import json
import google.generativeai as genai
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
def safe_generate_content(model, prompt):
    """Retries generation if quota is hit."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return model.generate_content(prompt)
        except exceptions.ResourceExhausted:
            wait_time = 30 * (attempt + 1)
            print(f"Quota exceeded. Waiting {wait_time}s before retry...")
            time.sleep(wait_time)
        except Exception as e:
            print(f"Gen Error: {e}")
            return None
    return None

def safe_embed_content(model, content, task_type):
    """Retries embedding if quota is hit."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return genai.embed_content(model=model, content=content, task_type=task_type)
        except exceptions.ResourceExhausted:
            wait_time = 30 * (attempt + 1)
            print(f"Quota exceeded. Waiting {wait_time}s before retry...")
            time.sleep(wait_time)
        except Exception as e:
            print(f"Embed Error: {e}")
            return None
    return None

In [ ]:
product_schema = {
    "type": "OBJECT",
    "properties": {
        "category": {
            "type": "ARRAY",  
            "items": {        
                "type": "STRING",
                "enum": ["brewed", "cold_brew", "espresso", "frappuccino", "refresher", "tea"]
            },
            "description": "List of categories. If user says 'coffee', include ['brewed', 'espresso', 'cold_brew']."
        },
        "temperature": {
            "type": "STRING",
            "enum": ["hot", "iced", "blended"]
        },
        "caffeine_level": {
            "type": "STRING",
            "enum": ["none", "low", "medium", "high"]
        },
        "max_calories": {"type": "INTEGER"},
        "max_sugar": {"type": "INTEGER"},
        "max_price": {"type": "NUMBER"},
        "dairy_free": {"type": "BOOLEAN"},
        "vegan": {"type": "BOOLEAN"}
    }
}

def extract_constraints(query_text: str) -> dict:
    
    
    system_instruction = """
    You are an expert Starbucks Barista AI. Extract structured search constraints.
    
    CRITICAL RULES:
    1. Output strictly valid JSON.
    2. ONLY include fields that are EXPLICITLY mentioned. OMIT unused keys.
    3. **CATEGORY MAPPING**:
       - If user says "coffee" (generic), return ALL coffee categories: ["brewed", "cold_brew", "espresso"].
       - If user says "tea", return ["tea"].
       - If user specifies "latte" or "cappuccino", return ["espresso"].
    4. Map 'frozen' -> temperature: 'blended'.
    5. Map 'decaf' -> caffeine_level: 'none'.
    """

    generation_config = {
        "temperature": 0.0,
        "response_mime_type": "application/json",
        "response_schema": product_schema
    }

    try:
        
        model = genai.GenerativeModel(
            model_name="gemini-2.5-flash", 
            system_instruction=system_instruction,
            generation_config=generation_config
        )
        response = model.generate_content(query_text)
        raw_result = json.loads(response.text)
        
        
        final_result = {
            "category": raw_result.get("category"), 
            "temperature": raw_result.get("temperature"),
            "caffeine_level": raw_result.get("caffeine_level"),
            "max_calories": raw_result.get("max_calories"),
            "max_sugar": raw_result.get("max_sugar"),
            "max_price": raw_result.get("max_price"),
            "dairy_free": raw_result.get("dairy_free"),
            "vegan": raw_result.get("vegan")
        }
        
        return final_result

    except Exception as e:
        print(f"Error: {e}")
        return {}


"""
if __name__ == "__main__":
    queries = [
        "Do you have a decaf iced tea?",
        "I want a coffee under 200 calories" 
    ]

    for q in queries:
        print(f"Query: {q}")
        print(f"Result: {extract_constraints(q)}")
        print("-" * 50)
"""

## Stage 2: Filter Products

In [ ]:
def load_and_preprocess_data(filepath: str) -> pd.DataFrame:
    """
    Loads product data and enriches it with the 'caffeine_level' column.
    """
    df = pd.read_csv(filepath)
    
    # Define logic to map caffeine_mg to categorical levels
    # Levels: None (0-15), Low (15-70), Medium (70-170), High (170+)
    conditions = [
        (df['caffeine_mg'] <= 15),
        (df['caffeine_mg'] > 15) & (df['caffeine_mg'] <= 70),
        (df['caffeine_mg'] > 70) & (df['caffeine_mg'] < 170),
        (df['caffeine_mg'] >= 170)
    ]
    choices = ['none', 'low', 'medium', 'high']
    
    # Create the new column
    df['caffeine_level'] = np.select(conditions, choices, default='medium')
    
    return df

# ==========================================
# 2. The Filtering Engine
# ==========================================
def filter_products(constraints: dict, products_df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters the product DataFrame based on the extracted constraints.
    Handles 'None' values by ignoring those filters.
    """
    # Start with the full list of products
    df = products_df.copy()

    # --- 1. Category Filter (Supports List or String) ---
    if constraints.get('category'):
        cat = constraints['category']
        if isinstance(cat, list):
            # If "coffee" -> ['brewed', 'espresso', ...], use ISIN
            df = df[df['category'].isin(cat)]
        else:
            # Single category match
            df = df[df['category'] == cat]

    # --- 2. Temperature Filter ---
    if constraints.get('temperature'):
        df = df[df['temperature'] == constraints['temperature']]

    # --- 3. Caffeine Level Filter ---
    if constraints.get('caffeine_level'):
        df = df[df['caffeine_level'] == constraints['caffeine_level']]

    # --- 4. Numerical Filters (Max Limits) ---
    # Only filter if the constraint is NOT None
    if constraints.get('max_price') is not None:
        df = df[df['price'] <= constraints['max_price']]

    if constraints.get('max_calories') is not None:
        df = df[df['calories'] <= constraints['max_calories']]

    if constraints.get('max_sugar') is not None:
        df = df[df['sugar_g'] <= constraints['max_sugar']]

    # --- 5. Dietary Filters (Boolean) ---
    # If dairy_free is True, we only want items where contains_dairy == False
    if constraints.get('dairy_free') is True:
        df = df[df['contains_dairy'] == False]

    # If vegan is True, we only want items where is_vegan == True
    if constraints.get('vegan') is True:
        df = df[df['is_vegan'] == True]

    return df

# ==========================================
# 3. Example Usage
# ==========================================
if __name__ == "__main__":
    # Load data (assuming file exists)
    try:
        products = load_and_preprocess_data('products.csv')
        
        # Scenario: User asked "I want a strong coffee under $5"
        # The LLM Stage 1 returns this:
        extracted_constraints = {
            'category': ['brewed', 'cold_brew', 'espresso'], # List of categories
            'temperature': None,
            'caffeine_level': 'high',    # "Strong"
            'max_price': 5.0,            # "Under $5"
            'max_calories': None,
            'max_sugar': None,
            'dairy_free': None,
            'vegan': None
        }

        print("Applying filters...")
        results = filter_products(extracted_constraints, products)
        
        print(f"Found {len(results)} matching products.")
        if not results.empty:
            print(results[['name', 'category', 'caffeine_level', 'price']].head())
            
    except FileNotFoundError:
        print("Error: 'products.csv' file not found. Please ensure the file is in the directory.")

## Stage 3: Rank by Relevance

In [ ]:
class GeminiRanker:
    def __init__(self):
        self.products_df = None
        self.product_embeddings = None

    def fit(self, df: pd.DataFrame):
        self.products_df = df
        # Create rich text for embedding
        self.products_df['embed_text'] = (
            "Product: " + self.products_df['name'].fillna('') + "; " +
            "Category: " + self.products_df['category'].fillna('') + "; " +
            "Desc: " + self.products_df['description'].fillna('')
        )
        
        print(f"Generating embeddings for {len(df)} products...")
        # Batch processing to avoid hitting API limits
        batch_size = 50 
        embeddings = []
        texts = self.products_df['embed_text'].tolist()
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i+batch_size]
            try:
                result = genai.embed_content(
                    model="models/text-embedding-004",
                    content=batch,
                    task_type="retrieval_document"
                )
                embeddings.extend(result['embedding'])
                time.sleep(0.5) # Rate limit safety
            except Exception as e:
                print(f"Embedding Error at batch {i}: {e}")
                # Fallback: append zeros if fail
                embeddings.extend([[0]*768] * len(batch))
                
        self.product_embeddings = np.array(embeddings)
        print("Embeddings ready.")

    def rank(self, query: str, candidate_df: pd.DataFrame, top_k=5) -> list:
        if candidate_df.empty:
            return []

        # 1. Embed Query
        try:
            q_res = genai.embed_content(
                model="models/text-embedding-004",
                content=query,
                task_type="retrieval_query"
            )
            q_vec = np.array([q_res['embedding']])
        except:
            return candidate_df['product_id'].head(top_k).tolist()

        # 2. Get Candidate Vectors
        indices = candidate_df.index.tolist()
        cand_vecs = self.product_embeddings[indices]

        # 3. Cosine Similarity
        scores = cosine_similarity(q_vec, cand_vecs).flatten()

        # 4. Sort
        results = candidate_df.copy()
        results['score'] = scores
        results = results.sort_values('score', ascending=False)
        
        return results['product_id'].head(top_k).tolist()

## Evaluation

In [ ]:
def calculate_ndcg(predicted_ids: list, true_ids_str: str, k=5) -> float:
    """
    Computes NDCG@k for a single query.
    Assumes true_ids_str is an ordered string like "ID1;ID2;ID3"
    """
    if not isinstance(true_ids_str, str) or not true_ids_str:
        return 0.0
    
    true_ids = [x.strip() for x in true_ids_str.split(';') if x.strip()]
    if not true_ids:
        return 0.0

    # Assign relevance scores: First item = len, Last item = 1
    # Ground Truth Relevance
    relevance_map = {pid: (len(true_ids) - i) for i, pid in enumerate(true_ids)}
    
    # Construct y_true (Ideal ranking scores) and y_score (Predicted scores)
    # We use the 'relevance' of the predicted items as their score
    y_true = np.array([[relevance_map[pid] for pid in true_ids]])
    
    # For predicted items, check their relevance score. If not in true list, score is 0.
    y_pred_relevance = [relevance_map.get(pid, 0) for pid in predicted_ids]
    
    # Pad if prediction is shorter than k
    if len(y_pred_relevance) < k:
        y_pred_relevance += [0] * (k - len(y_pred_relevance))
    
    # NDCG needs a 2D array: shape (n_samples, n_labels)
    # Since libraries calculate based on scores, we can simulate it:
    # However, simpler implementation for ordered lists:
    return ndcg_score([y_true[0][:k]], [y_pred_relevance[:k]], k=k)

## Train Execution

In [ ]:
def parse_relevant_products(row_value):
    """
    Parses the 'relevant_products' field from string to list.
    """
    if isinstance(row_value, list):
        return row_value
    if isinstance(row_value, str):
        try:
            return ast.literal_eval(row_value)
        except:
            return []
    return []

if __name__ == "__main__":
    try:
        # 1. Load Data
        products_df = pd.read_csv("products.csv")
        
        # Preprocess caffeine
        conds = [(products_df['caffeine_mg']<=15), (products_df['caffeine_mg']<=70), (products_df['caffeine_mg']<170)]
        choices = ['none', 'low', 'medium']
        products_df['caffeine_level'] = np.select(conds, choices, default='high')
        
        queries_train = pd.read_csv("queries_train.csv")
        
        # --- APPLY THE FIX ---
        queries_train['relevant_products'] = queries_train['relevant_products'].apply(parse_relevant_products)
        
    except FileNotFoundError:
        print("Error: Files not found.")
        exit()

    # 2. Fit Ranker
    ranker = GeminiRanker()
    ranker.fit(products_df)

    # 3. Validation Loop
    ndcg_scores = []
    print(f"\nValidating on {len(queries_train)} queries (this may take time due to rate limits)...")

    for idx, row in queries_train.iterrows():
        qid = row['query_id']
        query = row['query_text']
        true_ids = row['relevant_products']
        
        # A. Extract
        cons = extract_constraints(query)
        
        # B. Filter
        cands = filter_products(cons, products_df)
        
        # Fallback
        if cands.empty:
            relaxed = {k:v for k,v in cons.items() if k in ['category', 'dairy_free', 'vegan']}
            cands = filter_products(relaxed, products_df)
            if cands.empty: cands = products_df
            
        # C. Rank
        pred_ids = ranker.rank(query, cands, top_k=5)
        
        # D. Calculate NDCG
        if true_ids:
            # Create relevance map (1st item = highest score)
            relevance_map = {pid: (len(true_ids)-i) for i, pid in enumerate(true_ids)}
            
            y_true = [relevance_map[pid] for pid in true_ids]
            y_pred = [relevance_map.get(pid, 0) for pid in pred_ids]
            
            # Pad to k=5
            k = 5
            y_true = (y_true + [0]*k)[:k]
            y_pred = (y_pred + [0]*k)[:k]
            
            if sum(y_true) > 0:
                score = ndcg_score([y_true], [y_pred], k=k)
                ndcg_scores.append(score)
                print(f"[{qid}] NDCG: {score:.4f}")
            else:
                ndcg_scores.append(0.0)
        
        # Important: Prevent 429 errors
        time.sleep(2)

    # 4. Final Output
    print("\n" + "="*40)
    if ndcg_scores:
        final_avg = np.mean(ndcg_scores)
        print(f"FINAL AVERAGE NDCG SCORE: {final_avg:.4f}")
    else:
        print("No scores calculated.")
    print("="*40)

## Test Execution

In [ ]:
if __name__ == "__main__":
    # Load Data
    products_df = pd.read_csv("products.csv")
    products_df = preprocess_products(products_df)
    
    # Check if we have validation labels
    queries_df = pd.read_csv("queries_test.csv")
    has_labels = 'relevant_products' in queries_df.columns and queries_df['relevant_products'].notna().any()
    
    # Initialize Ranker
    ranker = GeminiRanker()
    ranker.fit(products_df)

    ndcg_scores = []
    results_csv = []

    print(f"\nProcessing {len(queries_df)} queries...")
    
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        query = row['query_text']
        
        # 1. Extract
        constraints = extract_constraints(query)
        
        # 2. Filter
        candidates = filter_products(constraints, products_df)
        
        # Fallback: Relax constraints if empty
        if candidates.empty:
            # Drop numeric constraints, keep category/dietary
            relaxed = {k:v for k,v in constraints.items() if k in ['category', 'dairy_free', 'vegan']}
            candidates = filter_products(relaxed, products_df)
            if candidates.empty:
                candidates = products_df # Last resort: Search all

        # 3. Rank
        top_products = ranker.rank(query, candidates, top_k=5)
        
        # 4. Score (if labels exist)
        if has_labels:
            true_products = row['relevant_products']
            score = calculate_ndcg(top_products, true_products, k=5)
            ndcg_scores.append(score)
            print(f"[{qid}] NDCG: {score:.4f} | Found: {len(candidates)}")
        else:
            print(f"[{qid}] Processed (No labels)")

        results_csv.append({"query_id": qid, "products": ";".join(top_products)})

    # Final Report
    if ndcg_scores:
        avg_ndcg = np.mean(ndcg_scores)
        print("="*40)
        print(f"FINAL AVERAGE NDCG@5 SCORE: {avg_ndcg:.4f}")
        print("="*40)

    # Save Submission
    pd.DataFrame(results_csv).to_csv("submission_v1.csv", index=False)
    print("Results saved to submission_v1.csv")